# Link Prediction mit GATv2

Dieses Notebook enthält **nur** Modell, Training und Auswertung. Die Daten kommen aus
`processed/philadelphia_split.pt`, erzeugt durch `data_prep.ipynb` (dort auch die Feature-Konfiguration).
Trainingsloop, Eingangsschicht, Decoder und Hyperparameter sind identisch zu `GINE.ipynb`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from gnn_pipeline import (
    load_bundle, set_seed, EdgeAwareInput, LinkDecoder, batch_norm, count_parameters,
    train_link_predictor, evaluate_link_predictor, run_seeds, plot_learning_curves, auc_by_link_type,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

bundle = load_bundle(device=device)   # processed/philadelphia_split.pt
data, train_data, val_data, test_data = bundle.data, bundle.train, bundle.val, bundle.test

in_channels = data.x.size(1)
edge_dim = data.edge_attr.size(1)

## Graph Attention Network v2 (GATv2) mit Kantenfeatures

GATv2 verbessert GAT, indem die Attention-Gewichte nach der Transformation der Nachbarfeatures
berechnet werden („dynamic attention“):

$$\alpha_{ij} = \mathrm{softmax}_j\Big(a^\top \mathrm{LeakyReLU}\big(W_l x_j + W_r x_i + W_e e_{ji}\big)\Big), \qquad x_i' = \sum_j \alpha_{ij}\, W_l x_j$$

Über `edge_dim` fließen die Kantenfeatures (`Capacity`, `Length`, `Toll`, `free_flow_time`, …) in die
Attention-Gewichte ein – anders als bei GINE nicht in die Nachricht selbst. Die Aggregation ist ein
gewichteter Mittelwert, der Knotengrad geht also nicht direkt in die Größe des Embeddings ein.

**Architektur:** gemeinsame Eingangsschicht `EdgeAwareInput` (Koordinaten + Summe der eingehenden
Kantenfeatures) → 3 × `GATv2Conv` (4, 4, 1 Heads; 32 Kanäle pro Head, also 128 nach dem Concat) mit
BatchNorm (ohne Running-Stats), ELU und Dropout → Knoten-Embeddings → gemeinsamer `LinkDecoder` auf
`concat(src, dst)`.

**Warum die Eingangsschicht:** Mit nur `x, y` als Knotenfeature sind die Koordinaten benachbarter Knoten
fast identisch. Der Attention-Mittelwert liefert dann kaum mehr als die eigene Position; die Kantenfeatures
wirken in `GATv2Conv` nur auf die Gewichte, nicht auf die Nachricht, und der Knotengrad geht durch die
Softmax-Normierung verloren. Ohne Eingangsschicht blieb GATv2 deshalb bei AUC 0.5.

**Warum nicht mehr 8 × 512 Kanäle:** Die frühere Variante hatte effektiv 4096 Kanäle und 36 Mio. Parameter,
ohne Normalisierung, mit ReLU und lr = 0.005. Schon der erste Adam-Schritt ließ die Aktivierungen explodieren
(Val-Loss > 100 in Epoche 1), danach waren fast alle ReLUs tot und das Modell sagte ~160 Epochen lang nur die
Basisrate voraus (Loss 0.6365 = Entropie bei 1/3 Positiven). Kleinere Breite, BatchNorm, ELU, LR-Warmup und
Gradient Clipping verhindern diesen Kollaps.

In [ ]:
from torch_geometric.nn import GATv2Conv

class GATv2LinkPredictor(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, edge_dim, heads=4, dropout=0.3):
        super().__init__()
        if hidden_channels % heads:
            raise ValueError("hidden_channels muss durch heads teilbar sein.")
        per_head = hidden_channels // heads   # concat der Heads ergibt wieder hidden_channels
        # gemeinsame Eingangsschicht (identisch zu GINE): Koordinaten + Summe der eingehenden Kantenfeatures
        self.input = EdgeAwareInput(in_channels, edge_dim, hidden_channels)
        self.conv1 = GATv2Conv(hidden_channels, per_head, heads=heads, edge_dim=edge_dim, concat=True)
        self.conv2 = GATv2Conv(hidden_channels, per_head, heads=heads, edge_dim=edge_dim, concat=True)
        self.conv3 = GATv2Conv(hidden_channels, out_channels, heads=1, edge_dim=edge_dim, concat=False)
        self.bn0 = batch_norm(hidden_channels)
        self.bn1 = batch_norm(hidden_channels)
        self.bn2 = batch_norm(hidden_channels)
        self.dropout = nn.Dropout(dropout)
        self.decoder = LinkDecoder(out_channels, hidden_channels, dropout)   # identisch zu GINE

    def forward(self, x, edge_index, edge_attr):
        # ELU wie im Original-GAT: kann im Gegensatz zu ReLU nicht "sterben"
        x = F.elu(self.bn0(self.input(x, edge_index, edge_attr)))
        x = self.dropout(F.elu(self.bn1(self.conv1(x, edge_index, edge_attr))))
        x = self.dropout(F.elu(self.bn2(self.conv2(x, edge_index, edge_attr))))
        return self.conv3(x, edge_index, edge_attr)

    def predict_link(self, node_embeddings, edge_label_index):
        return self.decoder(node_embeddings, edge_label_index)

In [ ]:
MODEL_NAME = "GATv2"
# Identisch in GINE.ipynb und GATV2.ipynb, damit sich die Modelle nur im Encoder unterscheiden
HIDDEN, OUT, DROPOUT = 128, 128, 0.3
EPOCHS, LR, WEIGHT_DECAY = 300, 0.002, 5e-4
SEED = bundle.meta.get("seed", 42)
HEADS = 4                      # conv1 und conv2; conv3 hat immer 1 Head

def make_model():
    return GATv2LinkPredictor(in_channels, HIDDEN, OUT, edge_dim, heads=HEADS, dropout=DROPOUT)

set_seed(SEED)
model = make_model().to(device)
print(model)
print(f"Parameter: {count_parameters(model):,}")

In [ ]:
TRAIN_KWARGS = dict(epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY)

history = train_link_predictor(model, train_data, val_data, test_data, log_every=10, **TRAIN_KWARGS)

# model hat jetzt die Gewichte der Epoche mit der besten Val-AUC
test_loss, test_auc = evaluate_link_predictor(model, test_data)
print(f"Test (Modell der besten Val-Epoche): Loss {test_loss:.4f} | ROC-AUC {test_auc:.4f}")

## Learning Curve

`plot_learning_curves` (in `gnn_pipeline/training.py`) zeigt die komplette `history`: links den BCE-Loss,
rechts die ROC-AUC, jeweils für Train, Val und Test. Alle drei Kurven sind im eval-Modus (ohne Dropout) auf
**demselben** Message-Passing-Graphen gemessen; keine der bewerteten Kanten liegt selbst im Graphen. Die
Kurven sind damit direkt vergleichbar. Die graue Linie markiert die Epoche mit der besten Val-AUC; auf diese
Gewichte wird das Modell nach dem Training zurückgesetzt, alle folgenden Zellen nutzen also dieses Modell.

**Worauf man achten sollte:**
- **Train über Val/Test:** Ein kleiner, stabiler Abstand ist normal (das Modell hat die Trainingspaare
  gesehen). Wächst er, während die Val-AUC stagniert, ist das Overfitting – das Checkpointing fängt es ab.
- **Val und Test liegen aufeinander:** erwartet, beide Splits sind gleich gezogen und sehen denselben Graphen.
- **Einordnung:** Die Heuristik-Baselines in `data_prep.ipynb` (Luftlinie ≈ 0.72, Grad ≈ 0.53) sind die
  Messlatte. Alles darüber hat das Modell aus der Graphstruktur gelernt.
- **Einzelner Lauf:** Eine Kurve ist eine Stichprobe. Für Aussagen über das Modell zählt die
  Mehr-Seed-Auswertung unten.

In [ ]:
plot_learning_curves(history, title=f"{MODEL_NAME}: Learning Curve ({len(history['val_auc'])} Epochen, Full-Batch, Seed {SEED})")
plt.show()

## Mehr-Seed-Auswertung

Einzelne Läufe streuen (Initialisierung, Dropout, nicht-deterministische GPU-Kernels), frühere
Läufe mit identischem Setup lagen bis zu ~0.015 AUC auseinander. Für den Vergleich GINE ↔ GATv2 zählt daher
Mittelwert ± Standardabweichung über mehrere Seeds, jeweils an der Epoche mit der besten Val-AUC (Auswahl auf
Val, nicht auf Test). Die Ergebnisse werden nach `processed/seeds_<Modell>.csv` geschrieben.

Laufzeit: pro Seed etwa so lange wie das Training oben.

In [ ]:
SEEDS = [0, 1, 2, 3, 4]
seed_df, seed_histories = run_seeds(make_model, train_data, val_data, test_data, SEEDS, **TRAIN_KWARGS)
seed_df.to_csv(f"processed/seeds_{MODEL_NAME}.csv", index=False)
seed_df

## Visualisierung der Knoten-Embeddings

PCA der gelernten Embeddings (Modell der besten Val-Epoche, vollständiger Graph) auf 2 Dimensionen,
eingefärbt nach Knotentyp (Zone / Kreuzung laut TNTP-Dateikopf).

In [ ]:
from sklearn.decomposition import PCA

model.eval()
with torch.no_grad():
    emb = model(data.x, data.edge_index, data.edge_attr).cpu().numpy()

emb_2d = PCA(n_components=2).fit_transform(emb)
plot_df = pd.DataFrame({'component_1': emb_2d[:, 0], 'component_2': emb_2d[:, 1], 'node_type': bundle.node_type})

plt.figure(figsize=(12, 10))
sns.scatterplot(data=plot_df, x='component_1', y='component_2', hue='node_type', palette='viridis', alpha=0.7, s=20)
plt.title(f'2D PCA of {MODEL_NAME} Node Embeddings (colored by node type)', size=16)
plt.xlabel('Principal Component 1'); plt.ylabel('Principal Component 2')
plt.legend(title='Node Type'); plt.grid(True, linestyle='--', alpha=0.6); plt.tight_layout()
plt.show()

## Fehleranalyse: Welche Straßentypen sagt das Modell gut vorher?

Für jeden Link-Typ: ROC-AUC der positiven Val-Kanten dieses Typs gegen alle negativen Val-Paare, dazu die
mittlere vorhergesagte Wahrscheinlichkeit. Typ 7 mit Kapazität 999999 sind die künstlichen
Zonen-Konnektoren, keine echten Straßen.

(Die frühere Korrelation „Kantenfeature ↔ Label“ war konstruktionsbedingt ≈ 0, weil Negative zufällig
gezogene Kantenfeatures tragen, und wurde deshalb ersetzt.)

In [ ]:
link_df = auc_by_link_type(model, val_data, bundle.edge_attr_columns)
display(link_df)

plt.figure(figsize=(10, 5))
sns.barplot(x=link_df.index, y=link_df["ROC-AUC vs. alle Negativen"], color="#2a78d6")
plt.ylim(max(0.0, link_df["ROC-AUC vs. alle Negativen"].min() - 0.05), 1.0)
plt.title(f"{MODEL_NAME}: Val-ROC-AUC nach Link-Typ der positiven Kante", size=14)
plt.xlabel("Link-Typ"); plt.ylabel("ROC-AUC"); plt.grid(axis="y", linestyle="--", alpha=0.6); plt.tight_layout()
plt.show()